<a href="https://colab.research.google.com/github/Shikher-jain/Data_Science/blob/main/DL/PYTORCH/CampusX_PT/cnn_fashion_mnist_pytorch_gpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [ ]:
# Set random seeds for reproducibility
torch.manual_seed(42)

In [ ]:
# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# 'https://www.kaggle.com/datasets/zalando-research/fashionmnist/data?select=fashion-mnist_train.csv'
df = pd.read_csv('fashion-mnist_train.csv')
df.head()

In [ ]:
df.shape

In [ ]:
# Create a 4x4 grid of images
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
fig.suptitle("First 16 Images", fontsize=16)

# Plot the first 16 images from the dataset
for i, ax in enumerate(axes.flat):
    img = df.iloc[i, 1:].values.reshape(28, 28)  # Reshape to 28x28
    ax.imshow(img)  # Display in grayscale
    ax.axis('off')  # Remove axis for a cleaner look
    ax.set_title(f"Label: {df.iloc[i, 0]}")  # Show the label

plt.tight_layout(rect=[0, 0, 1, 0.96])  # Adjust layout to fit the title
plt.show()


In [ ]:
# train test split

X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values

In [80]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [81]:
X_train = X_train/255.0
X_test = X_test/255.0

In [82]:
class CustomDataset(Dataset):

  def __init__(self, features, labels):

    # Convert to PyTorch tensors
    self.features = torch.tensor(features, dtype=torch.float32)
    self.labels = torch.tensor(labels, dtype=torch.long)

  def __len__(self):
    return len(self.features)

  def __getitem__(self, index):
    return self.features[index], self.labels[index]

In [83]:
train_dataset = CustomDataset(X_train, y_train)

In [84]:
test_dataset = CustomDataset(X_test, y_test)

In [85]:
class MyNN(nn.Module):

  def __init__(self, input_dim,output_dim,num_hidden_layers, neurons_per_layer,dropout_rate):

    super().__init__()
    layers = []

    for i in range(num_hidden_layers):
        layers.append(nn.Linear(input_dim, neurons_per_layer))
        input_dim = neurons_per_layer

        layers.append(nn.BatchNorm1d(neurons_per_layer))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(p=dropout_rate))

    layers.append(nn.Linear(neurons_per_layer, output_dim))

    self.model = nn.Sequential(*layers)

  def forward(self, x):
    return self.model(x)


In [86]:
#Objective Function

def objective(trial):

    # next hyperparameter values from the search space
    num_hidden_layers = trial.suggest_int('num_hidden_layers', 1, 5)
    num_hidden_units = trial.suggest_int('num_hidden_units', 8, 128, step =8)

    # params init
    epochs = trial.suggest_int('epochs', 10, 50,step=10)
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-1, log=True)

    dropout_rate = trial.suggest_float('dropout_rate', 0.0, 0.5, step=0.1)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64, 128])
    weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-1, log=True)

    optimizer = trial.suggest_categorical('optimizer', ['Adam', 'SGD', 'RMSprop'])

    # dataloader init
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)

    # model init
    input_dim = 784
    output_dim = 10

    model = MyNN(input_dim, output_dim, num_hidden_layers, num_hidden_units,dropout_rate)
    model.to(device)



    # optimizer SelectionMixin
    criterion = nn.CrossEntropyLoss()

    if optimizer == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    elif optimizer == 'RMSprop':
        optimizer = optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    elif optimizer == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    optimizer = optim.SGD(model.parameters(), lr=learning_rate,weight_decay=weight_decay)

    # training Loop

    for epoch in range(epochs):
        for batch_features, batch_labels in train_loader:

            # move data to gpu
            batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

            # forward pass
            outputs = model(batch_features)

            # calculate loss
            loss = criterion(outputs, batch_labels)

            # back pass
            optimizer.zero_grad()
            loss.backward()

            # update grads
            optimizer.step()

    # evaluation
    model.eval()

    total = 0
    correct = 0

    with torch.no_grad():

        for batch_features, batch_labels in test_loader:

            # move data to gpu
            batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

            outputs = model(batch_features)

            _, predicted = torch.max(outputs, 1)

            total = total + batch_labels.shape[0]

            correct = correct + (predicted == batch_labels).sum().item()

        print(accuracy := correct/total)
    return accuracy

In [87]:
# !pip install optuna

In [90]:
import optuna
import time

start = time.time()
study = optuna.create_study(direction='maximize')
print("start")
study.optimize(objective, n_trials=10)
end = time.time()
print(f"Time taken: {end-start}")

[I 2026-05-28 17:33:27,867] A new study created in memory with name: no-name-db478cc7-1c8c-4c2c-8462-a67e3b2010b6


start


[I 2026-05-28 17:37:17,759] Trial 0 finished with value: 0.8259166666666666 and parameters: {'num_hidden_layers': 3, 'num_hidden_units': 88, 'epochs': 30, 'learning_rate': 6.37680210547597e-05, 'dropout_rate': 0.1, 'batch_size': 16, 'weight_decay': 0.0003515131687441023, 'optimizer': 'SGD'}. Best is trial 0 with value: 0.8259166666666666.


0.8259166666666666


[I 2026-05-28 17:38:10,872] Trial 1 finished with value: 0.29783333333333334 and parameters: {'num_hidden_layers': 5, 'num_hidden_units': 128, 'epochs': 10, 'learning_rate': 2.171170984121828e-05, 'dropout_rate': 0.4, 'batch_size': 32, 'weight_decay': 0.00047609556652125715, 'optimizer': 'SGD'}. Best is trial 0 with value: 0.8259166666666666.


...

0.5660833333333334


[I 2026-05-28 17:47:20,327] Trial 8 finished with value: 0.8673333333333333 and parameters: {'num_hidden_layers': 1, 'num_hidden_units': 88, 'epochs': 40, 'learning_rate': 0.003682397734032465, 'dropout_rate': 0.30000000000000004, 'batch_size': 128, 'weight_decay': 0.0032873270052455232, 'optimizer': 'RMSprop'}. Best is trial 8 with value: 0.8673333333333333.


0.8673333333333333


[I 2026-05-28 17:47:55,775] Trial 9 finished with value: 0.34675 and parameters: {'num_hidden_layers': 2, 'num_hidden_units': 8, 'epochs': 20, 'learning_rate': 5.482255870074238e-05, 'dropout_rate': 0.2, 'batch_size': 64, 'weight_decay': 0.000369581894063937, 'optimizer': 'SGD'}. Best is trial 8 with value: 0.8673333333333333.


0.34675
Time taken: 867.9104239940643


In [92]:
study.best_value

0.8673333333333333

In [94]:
study.best_params

{'num_hidden_layers': 1,
 'num_hidden_units': 88,
 'epochs': 40,
 'learning_rate': 0.003682397734032465,
 'dropout_rate': 0.30000000000000004,
 'batch_size': 128,
 'weight_decay': 0.0032873270052455232,
 'optimizer': 'RMSprop'}

In [95]:
study.best_trials

[FrozenTrial(number=8, state=<TrialState.COMPLETE: 1>, values=[0.8673333333333333], datetime_start=datetime.datetime(2026, 5, 28, 17, 46, 45, 669360), datetime_complete=datetime.datetime(2026, 5, 28, 17, 47, 20, 327675), params={'num_hidden_layers': 1, 'num_hidden_units': 88, 'epochs': 40, 'learning_rate': 0.003682397734032465, 'dropout_rate': 0.30000000000000004, 'batch_size': 128, 'weight_decay': 0.0032873270052455232, 'optimizer': 'RMSprop'}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'num_hidden_layers': IntDistribution(high=5, log=False, low=1, step=1), 'num_hidden_units': IntDistribution(high=128, log=False, low=8, step=8), 'epochs': IntDistribution(high=50, log=False, low=10, step=10), 'learning_rate': FloatDistribution(high=0.1, log=True, low=1e-05, step=None), 'dropout_rate': FloatDistribution(high=0.5, log=False, low=0.0, step=0.1), 'batch_size': CategoricalDistribution(choices=(16, 32, 64, 128)), 'weight_decay': FloatDistribution(high=0.1, log=Tru